# Code Complexity vs Agent Resolution Rate

**Question**: Does structural code complexity predict whether coding agents can fix bugs?

**Hypothesis**: Code that is hard for humans to maintain (high cyclomatic/cognitive complexity, low maintainability index) should also be hard for coding agents to fix.

**Finding**: File-level complexity metrics do **not** predict agent success. The further you zoom out from the actual change, the weaker the signal. Only patch-level metrics (what the agent must produce) correlate significantly.

We analyze metrics at three granularity levels:
1. **Patch-level** — size and scatter of the gold patch (LOC, files, hunks)
2. **Function-level** — complexity of the specific function(s) being modified
3. **File-level** — whole-file structural complexity (cyclomatic, Halstead, MI, etc.)

In [ ]:
import json
import subprocess
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from unidiff import PatchSet

sys.path.insert(0, str(Path.cwd().parent))

from utils import DATASET_PATH, count_files_in_patch, count_loc_in_patch, load_aggregate_results

from bcbench.dataset import DatasetEntry, load_dataset_entries

METRICS_PATH = DATASET_PATH.parent / "bcbench_metrics.jsonl"
FN_METRICS_PATH = DATASET_PATH.parent / "bcbench_fn_metrics.json"
ANALYSIS_SETUP = "copilot-opus-4-6"

## 1. Load Data

In [ ]:
# --- Resolution rates (multiple runs) ---
all_results = load_aggregate_results(category="bug-fix")
runs_df = all_results[ANALYSIS_SETUP]

resolution_df = (
    runs_df.groupby("instance_id")["resolved"]
    .agg(resolution_rate="mean", n_runs="count", n_resolved="sum")
    .reset_index()
)

print(f"Setup: {ANALYSIS_SETUP}")
print(f"Instances: {len(resolution_df)}, Runs per instance: {resolution_df['n_runs'].iloc[0]}")
print(f"Overall resolution rate: {resolution_df['resolution_rate'].mean():.1%}")

In [ ]:
# --- Dataset + patch-level features ---
dataset_entries = load_dataset_entries(DATASET_PATH)
entries_raw = [json.loads(l) for l in DATASET_PATH.read_text(encoding="utf-8").splitlines()]

patch_rows = []
for e_raw, e in zip(entries_raw, dataset_entries):
    ps = PatchSet(e_raw["patch"])
    patch_rows.append({
        "instance_id": e.instance_id,
        "area": e.metadata.area or "Unknown",
        "project": e.extract_project_name(),
        "patch_loc": count_loc_in_patch(e.patch),
        "patch_files": count_files_in_patch(e.patch),
        "total_hunks": sum(len(list(pf)) for pf in ps),
    })
patch_df = pd.DataFrame(patch_rows)

# --- File-level metrics (rust-code-analysis, whole file) ---
file_rows = []
for line in METRICS_PATH.read_text(encoding="utf-8").splitlines():
    entry = json.loads(line)
    all_m = [f["metrics"] for f in entry["files"] if f["metrics"]]
    if not all_m:
        file_rows.append({"instance_id": entry["instance_id"]})
        continue

    total_sloc = sum(m["loc"]["sloc"] for m in all_m) or 1
    patch_lines = sum(
        pf.added + pf.removed
        for pf in PatchSet(
            next(e for e in entries_raw if e["instance_id"] == entry["instance_id"])["patch"]
        )
    )

    file_rows.append({
        "instance_id": entry["instance_id"],
        "file_cyclomatic": sum(m["cyclomatic"]["sum"] for m in all_m),
        "file_cyclomatic_max": max(m["cyclomatic"]["max"] for m in all_m),
        "file_cognitive": sum(m["cognitive"]["sum"] for m in all_m),
        "file_cognitive_max": max(m["cognitive"]["max"] for m in all_m),
        "file_sloc": sum(m["loc"]["sloc"] for m in all_m),
        "file_halstead_volume": sum(m["halstead"]["volume"] for m in all_m),
        "file_halstead_effort": sum(m["halstead"]["effort"] for m in all_m),
        "file_halstead_bugs": sum(m["halstead"]["bugs"] for m in all_m),
        "file_num_functions": sum(m["nom"]["functions"] for m in all_m),
        "file_mi_vs": sum(m["mi"]["mi_visual_studio"] * m["loc"]["sloc"] for m in all_m) / total_sloc,
        "file_abc_magnitude": sum(m["abc"]["magnitude"] for m in all_m),
        "patch_pct_of_file": patch_lines / total_sloc * 100,
    })
file_df = pd.DataFrame(file_rows).fillna(0)

# --- Function-level metrics ---
fn_df = pd.DataFrame(json.loads(FN_METRICS_PATH.read_text(encoding="utf-8")))

# --- Merge ---
df = (
    resolution_df
    .merge(patch_df, on="instance_id")
    .merge(file_df, on="instance_id")
    .merge(fn_df, on="instance_id", how="left")
)
print(f"Merged: {len(df)} instances ({df['fn_cyclomatic_sum'].notna().sum()} with function-level metrics)")
df.head(2)

## 2. The Granularity Mismatch

Before looking at correlations, let's understand **why** file-level metrics might fail: patches typically touch a tiny fraction of the file.

In [ ]:
pct = df["patch_pct_of_file"]
print("Patch as % of file SLOC:")
print(f"  Median: {pct.median():.1f}%")
print(f"  Mean:   {pct.mean():.1f}%")
print(f"  < 1%:   {(pct < 1).sum()} / {len(pct)} ({(pct < 1).mean():.0%})")
print(f"  < 5%:   {(pct < 5).sum()} / {len(pct)} ({(pct < 5).mean():.0%})")
print(f"  < 10%:  {(pct < 10).sum()} / {len(pct)} ({(pct < 10).mean():.0%})")

fig = px.histogram(
    df, x="patch_pct_of_file", nbins=50,
    labels={"patch_pct_of_file": "Patch as % of File SLOC"},
    title="Distribution: How much of the file does each gold patch touch?",
)
fig.add_vline(x=1, line_dash="dash", line_color="red", annotation_text="1%")
fig.add_vline(x=5, line_dash="dash", line_color="orange", annotation_text="5%")
fig.update_layout(height=350, width=700, showlegend=False)
fig.show()

## 3. Three-Tier Correlation Analysis

We compare Spearman rank correlations at three granularity levels:
- **Patch-level**: properties of the gold patch itself
- **Function-level**: complexity of the specific function(s) being modified
- **File-level**: whole-file structural metrics

Spearman is used because it doesn't assume linearity and is robust to outliers.

In [ ]:
TIERS = [
    ("Patch-level", [
        ("total_hunks", "Number of Hunks"),
        ("patch_loc", "Lines Changed (LOC)"),
        ("patch_files", "Files Changed"),
        ("patch_pct_of_file", "Patch % of File"),
    ]),
    ("Function-level", [
        ("fn_cyclomatic_sum", "Cyclomatic (patched fn)"),
        ("fn_cognitive_sum", "Cognitive (patched fn)"),
        ("fn_sloc_sum", "SLOC (patched fn)"),
        ("fn_halstead_volume_sum", "Halstead Volume (patched fn)"),
        ("fn_halstead_effort_sum", "Halstead Effort (patched fn)"),
    ]),
    ("File-level", [
        ("file_cyclomatic", "Cyclomatic (whole file)"),
        ("file_cognitive", "Cognitive (whole file)"),
        ("file_sloc", "SLOC (whole file)"),
        ("file_halstead_volume", "Halstead Volume (whole file)"),
        ("file_halstead_effort", "Halstead Effort (whole file)"),
        ("file_halstead_bugs", "Halstead Est. Bugs (whole file)"),
        ("file_mi_vs", "Maintainability Index (whole file)"),
        ("file_num_functions", "Num Functions (whole file)"),
        ("file_abc_magnitude", "ABC Magnitude (whole file)"),
    ]),
]

corr_rows = []
for tier_name, metrics in TIERS:
    for col, label in metrics:
        valid = df[[col, "resolution_rate"]].dropna()
        if len(valid) < 10:
            continue
        rho, p = stats.spearmanr(valid[col], valid["resolution_rate"])
        corr_rows.append({
            "tier": tier_name,
            "column": col,
            "metric": label,
            "spearman_rho": round(rho, 3),
            "p_value": round(p, 4),
            "significant": p < 0.05,
            "n": len(valid),
        })

corr_df = pd.DataFrame(corr_rows)

# Print grouped table
for tier in ["Patch-level", "Function-level", "File-level"]:
    subset = corr_df[corr_df["tier"] == tier].sort_values("p_value")
    print(f"\n{tier.upper()}")
    for _, r in subset.iterrows():
        sig = "**" if r["p_value"] < 0.01 else "* " if r["significant"] else "  "
        print(f"  {sig} {r['metric']:40s} ρ={r['spearman_rho']:+.3f}  p={r['p_value']:.4f}  (n={r['n']})")

In [ ]:
# Bar chart: correlation coefficients grouped by tier
tier_colors = {"Patch-level": "#2ecc71", "Function-level": "#f39c12", "File-level": "#e74c3c"}

plot_df = corr_df.sort_values("spearman_rho")

fig = go.Figure()
fig.add_trace(go.Bar(
    y=[f"{r['metric']}" for _, r in plot_df.iterrows()],
    x=plot_df["spearman_rho"],
    orientation="h",
    marker_color=[tier_colors[t] for t in plot_df["tier"]],
    text=[
        f"ρ={r['spearman_rho']:.3f} {'*' if r['significant'] else ''}" 
        for _, r in plot_df.iterrows()
    ],
    textposition="outside",
))

# Add invisible traces for legend
for tier, color in tier_colors.items():
    fig.add_trace(go.Bar(
        y=[None], x=[None], orientation="h",
        marker_color=color, name=tier, showlegend=True,
    ))

fig.update_layout(
    title=f"Spearman ρ with Resolution Rate by Metric Granularity ({ANALYSIS_SETUP})",
    xaxis_title="Spearman ρ (negative = higher metric → lower resolution)",
    height=650,
    width=950,
    margin=dict(l=300),
    xaxis=dict(range=[-0.45, 0.15], zeroline=True, zerolinecolor="black", zerolinewidth=1),
    showlegend=True,
    legend=dict(x=0.01, y=0.01),
)
fig.show()

## 4. Binned Resolution Rates — Significant Metrics

In [ ]:
def resolution_by_bins(df: pd.DataFrame, col: str, bins: list, labels: list, col_label: str) -> pd.DataFrame:
    binned = pd.cut(df[col], bins=bins, labels=labels)
    result = (
        df.assign(bin=binned)
        .groupby("bin", observed=True)
        .agg(resolution_rate=("resolution_rate", "mean"), count=("instance_id", "count"))
        .reset_index()
    )
    total = result["count"].sum()
    result["% Resolved"] = (result["resolution_rate"] * 100).round(1).astype(str) + "%"
    result["Instances"] = result["count"].astype(str) + " (" + (result["count"] / total * 100).round(1).astype(str) + "%)"
    return result.rename(columns={"bin": col_label})


# --- Number of Hunks (best single predictor) ---
hunk_bins = resolution_by_bins(df, "total_hunks", [0, 1, 2, 4, float("inf")], ["1", "2", "3-4", "5+"], "Hunks")
print("Resolution Rate by Number of Hunks (best predictor, ρ=-0.332):")
print(hunk_bins[["Hunks", "% Resolved", "Instances"]].to_string(index=False))

print()

# --- Patch LOC ---
loc_bins = resolution_by_bins(df, "patch_loc", [0, 5, 10, 50, float("inf")], ["1-5", "6-10", "11-50", "50+"], "Patch LOC")
print("Resolution Rate by Patch LOC (ρ=-0.323):")
print(loc_bins[["Patch LOC", "% Resolved", "Instances"]].to_string(index=False))

print()

# --- Files changed ---
files_bins = resolution_by_bins(df, "patch_files", [0, 1, 2, float("inf")], ["1", "2", "3+"], "Files Changed")
print("Resolution Rate by Files Changed (ρ=-0.266):")
print(files_bins[["Files Changed", "% Resolved", "Instances"]].to_string(index=False))

In [ ]:
# Bar chart for hunks (best predictor)
fig = px.bar(
    hunk_bins, x="Hunks", y="resolution_rate",
    text="% Resolved",
    labels={"resolution_rate": "Resolution Rate"},
    title="Resolution Rate by Number of Hunks (ρ=-0.332, p=0.0007)",
    color_discrete_sequence=["#2ecc71"],
)
fig.update_layout(height=350, width=500, yaxis=dict(range=[0, 1], tickformat=".0%"))
fig.show()

## 5. Scatter Plots — Top Predictors

In [ ]:
top_metrics = corr_df.reindex(corr_df["spearman_rho"].abs().sort_values(ascending=False).index).head(4)

for _, row in top_metrics.iterrows():
    col = row["column"]
    label = row["metric"]
    tier = row["tier"]
    rho = row["spearman_rho"]
    p = row["p_value"]

    valid = df[[col, "resolution_rate", "instance_id", "project", "area"]].dropna()
    fig = px.scatter(
        valid, x=col, y="resolution_rate",
        hover_data=["instance_id", "project", "area"],
        labels={col: label, "resolution_rate": "Resolution Rate"},
        title=f"[{tier}] {label} vs Resolution Rate (ρ={rho:.3f}, p={p:.4f})",
        trendline="lowess",
        trendline_color_override="red",
    )
    fig.update_layout(height=350, width=650, yaxis=dict(range=[-0.05, 1.05]))
    fig.show()

## 6. Cross-Setup Robustness

Do these patterns hold across different agent setups?

In [ ]:
all_agg = load_aggregate_results(category="bug-fix")

# Build per-instance feature table (no resolution, reused across setups)
feature_df = patch_df.merge(file_df, on="instance_id").merge(fn_df, on="instance_id", how="left")

# Key metrics to compare across setups
key_metrics = [
    ("total_hunks", "Hunks"),
    ("patch_loc", "Patch LOC"),
    ("patch_files", "Files Changed"),
    ("fn_cyclomatic_sum", "Cyclomatic (fn)"),
    ("fn_cognitive_sum", "Cognitive (fn)"),
    ("file_cyclomatic", "Cyclomatic (file)"),
    ("file_cognitive", "Cognitive (file)"),
    ("file_sloc", "SLOC (file)"),
    ("file_mi_vs", "MI (file)"),
]

cross_rows = []
for setup_name, setup_df in all_agg.items():
    setup_res = setup_df.groupby("instance_id")["resolved"].mean().reset_index().rename(columns={"resolved": "resolution_rate"})
    merged = setup_res.merge(feature_df, on="instance_id")
    for col, label in key_metrics:
        valid = merged[[col, "resolution_rate"]].dropna()
        if len(valid) < 10:
            continue
        rho, p = stats.spearmanr(valid[col], valid["resolution_rate"])
        cross_rows.append({"setup": setup_name, "metric": label, "spearman_rho": round(rho, 3), "p_value": p})

cross_df = pd.DataFrame(cross_rows)
pivot = cross_df.pivot_table(index="metric", columns="setup", values="spearman_rho")
pivot["avg_abs_rho"] = pivot.abs().mean(axis=1)
pivot = pivot.sort_values("avg_abs_rho", ascending=False).drop(columns="avg_abs_rho")

fig = go.Figure(data=go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale="RdBu",
    zmid=0, zmin=-0.4, zmax=0.4,
    text=[[f"{v:.3f}" for v in row] for row in pivot.values],
    texttemplate="%{text}",
    textfont=dict(size=12),
))
fig.update_layout(
    title="Code Metrics vs Resolution Rate — Cross-Setup Comparison",
    height=500, width=800, margin=dict(l=200),
    yaxis=dict(autorange="reversed"),
)
fig.show()

## 7. Summary

In [ ]:
tier_summary = corr_df.groupby("tier").agg(
    best_rho=("spearman_rho", lambda x: x.loc[x.abs().idxmax()]),
    best_metric=("metric", lambda x: x.loc[corr_df.loc[x.index, "spearman_rho"].abs().idxmax()]),
    n_significant=("significant", "sum"),
    n_total=("significant", "count"),
)

print("=" * 75)
print("SUMMARY: Three-Tier Correlation Analysis")
print("=" * 75)

for tier in ["Patch-level", "Function-level", "File-level"]:
    r = tier_summary.loc[tier]
    print(f"\n  {tier}:")
    print(f"    Best predictor:    {r['best_metric']} (ρ={r['best_rho']:.3f})")
    print(f"    Significant:       {r['n_significant']:.0f} / {r['n_total']:.0f} metrics")

print("\n" + "-" * 75)
print("\nKEY FINDING:")
print("  The further you zoom out from the actual change, the weaker the signal.")
print(f"  In {(df['patch_pct_of_file'] < 1).mean():.0%} of tasks, the patch touches <1% of the file.")
print("  File-level complexity is almost entirely noise relative to the bug fix.")
print("\n  Agent difficulty is dominated by the SEARCH problem (finding where to fix),")
print("  not by the structural complexity of the code being modified.")
print("\n  Number of hunks is the best single predictor — it captures 'scatter',")
print("  a proxy for how many distinct locations the agent must correctly modify.")